In [ ]:
!pip install -q -U google-genai

import os
import json
import glob
from kaggle_secrets import UserSecretsClient
from google import genai
from google.genai import types


In [ ]:
import os
import json
import glob
import time
from kaggle_secrets import UserSecretsClient # Import required

# --- CONFIGURATION ---
INPUT_DIR = "/kaggle/input/hunter-book-extraction/book_extracted_json"
# CHANGED: Use a .json extension for the pretty-printed array format
OUTPUT_FILE = "medical_finetune_dataset.json" 
CHAPTER_LIMIT = 67        # Set the maximum number of chapters to process
DELAY_SECONDS = 2        # Set the delay in seconds between each API call

# 1. Setup Client
try:
    user_secrets = UserSecretsClient()
    # Note: Corrected the secret label to be consistent with the print statement
    GEMINI_API_KEY = user_secrets.get_secret("gemini_api") 
    print("API key retrieved successfully.")
except Exception as e:
    print(f"Error: {e}")
    print("Make sure you added the secret 'gemini-api' in Add-ons -> Secrets")
    GEMINI_API_KEY = None # Handle the missing key case

# Ensure client is only initialized if key is available
if not GEMINI_API_KEY:
    raise EnvironmentError("Failed to retrieve GEMINI_API_KEY. Check your Kaggle secrets setup.")

from google import genai
from google.genai import types
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_NAME = "gemini-2.5-flash"

# ----------------------------------------
# --- GENERATION FUNCTION (Unchanged) ---
# ----------------------------------------

def generate_multi_disease_qa(chapter_data):
    """
    Generates 5 distinct Q&A pairs. If the chapter contains multiple diseases,
    it distributes the questions across them.
    """
    
    # 1. Prepare Context
    title = chapter_data.get("chapter_title", "Unknown")
    disease_label = chapter_data.get("disease_name", "Unknown")
    summaries = chapter_data.get("section_summaries", {})
    
    context_text = f"""
    Chapter Title: {title}
    Disease Scope: {disease_label}
    
    Epidemiology: {summaries.get('epidemiology', 'N/A')}
    Clinical Manifestations: {summaries.get('clinical_manifestations', 'N/A')}
    Diagnosis: {summaries.get('diagnosis', 'N/A')}
    Treatment: {summaries.get('treatment', 'N/A')}
    """
    
    # 2. Advanced Prompt for Multiple Diseases
    prompt = f"""
    You are an expert medical case writer creating a fine-tuning dataset.
    
    CONTEXT:
    {context_text}
    
    TASK:
    Generate exactly **5 distinct Instruction-Response pairs** based on the provided text.
    
    CRITICAL RULES FOR MULTIPLE DISEASES:
    - **Analyze the text first:** Does this chapter cover multiple different diseases or pathogens? (e.g. "Malaria" covers P. falciparum AND P. vivax; "Nematodes" covers Ascaris AND Hookworm).
    - **If YES:** You must generate separate patient cases for the distinct diseases. Do not focus all 5 questions on just one pathogen.
    - **If NO (Single disease):** Generate 5 distinct patient scenarios covering different presentations (e.g. uncomplicated vs. severe, child vs. adult, pregnant patient).
    
    STRUCTURE FOR EACH PAIR:
    Each of the 5 pairs must be a self-contained object with:
    1. **"instruction"**: A unique, synthesized **Patient Case (Vignette)** followed by a specific Question.
       - The Patient Case must be realistic (synthesize Age, Travel History, Symptoms based on Epidemiology/Clinical sections).
    2. **"response"**: The correct medical answer derived *strictly* from the provided Context.
    
    REQUIRED QUESTION TYPES (Distribute these among the 5 pairs):
    - **Diagnosis:** "What is the most likely diagnosis?" (Link specific symptoms to the disease).
    - **Differential:** "What is the key differential diagnosis?"
    - **Treatment (First-line):** "What is the primary treatment for this specific patient?"
    - **Treatment (Severe/Alternative):** "How should this severe/complicated case be managed?"
    - **Mechanism/Risk:** Question about risk factors or specific complications (e.g. "Why is this patient at risk for relapse?").
    
    OUTPUT FORMAT:
    Return a raw JSON list of 5 objects.
    Example:
    [
      {{"instruction": "Patient Case: A 30yo male returned from [Location]... \\n\\nQuestion: What is the diagnosis?", "response": "..."}},
      {{"instruction": "Patient Case: A 6yo child with [Specific Symptom]... \\n\\nQuestion: What is the treatment?", "response": "..."}}
    ]
    """
    
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            )
        )
        return json.loads(response.text)
    except Exception as e:
        print(f"Error processing {title}: {e}")
        return []

# ----------------------------------------
# --- MAIN EXECUTION LOOP (MODIFIED) ---
# ----------------------------------------

all_qa_pairs = []
json_files = glob.glob(os.path.join(INPUT_DIR, "*.json"))

print(f"Found {len(json_files)} files. Limiting processing to the first {CHAPTER_LIMIT} chapters found.")

# 1. Iterate and Generate
for i, json_file in enumerate(json_files):
    # Check the chapter limit
    if i >= CHAPTER_LIMIT:
        print(f"Reached the configured chapter limit of {CHAPTER_LIMIT}. Stopping processing.")
        break
        
    try:
        # Extract the base file name for tracking
        base_file_name = os.path.basename(json_file)
        
        with open(json_file, 'r') as f:
            data = json.load(f)
        
        d_name = data.get("disease_name", "Unknown")
        print(f"[{i+1}/{CHAPTER_LIMIT}] Generating 5 pairs for: {d_name} from file {base_file_name}...")
        
        pairs = generate_multi_disease_qa(data)
        
        if pairs:
            modified_pairs = []
            # Loop through generated pairs to add the source file name
            for pair in pairs:
                pair["source_file"] = base_file_name # NEW KEY ADDED HERE
                modified_pairs.append(pair)
                
            all_qa_pairs.extend(modified_pairs)
            print(f"   -> Success. Generated {len(modified_pairs)} pairs. Total gathered: {len(all_qa_pairs)}")
        else:
            print("   -> No pairs returned.")
            
    except Exception as e:
        print(f"Skipping file due to error: {e}")

    # 2. Add the wait time
    if i < CHAPTER_LIMIT - 1: # Only sleep if we have more chapters to process
        print(f"   -> Waiting {DELAY_SECONDS} seconds before starting the next chapter...")
        time.sleep(DELAY_SECONDS)

# ----------------------------------------
# 3. SAVE TO PRETTY-PRINTED JSON ARRAY
# ----------------------------------------
print(f"\nSaving all gathered data ({len(all_qa_pairs)} entries) to a pretty-printed JSON file...")

# Key Change: Dump the entire list, using 'indent=4' for readability
with open(OUTPUT_FILE, 'w') as f:
    json.dump(all_qa_pairs, f, indent=4) 

print(f"\nDone! Created {len(all_qa_pairs)} total training examples in {OUTPUT_FILE}")

In [ ]:
!zip -r book_extracted_json.zip book_extracted_json

# (Optional) zip other folders
!zip -r book_extracted_images.zip book_extracted_images
!zip -r book_extracted_tables.zip book_extracted_tables

In [ ]:
!zip -r temp_chapters.zip temp_chapters

In [ ]:
import os
import json
import glob
import time
from pathlib import Path
from PIL import Image
from kaggle_secrets import UserSecretsClient # Import required

# --- CONFIGURATION ---
IMAGE_INPUT_DIR = "/kaggle/input/d/miyachinenn/hunter-book-extraction/book_extracted_images/book_extracted_images"
JSON_INPUT_DIR = "/kaggle/input/d/miyachinenn/hunter-book-extraction/book_extracted_json"
OUTPUT_FILE = "medical_finetune_dataset_image.json"
PROGRESS_FILE = "image_processing_progress.json"  # Track state across kernel restarts
IMAGE_LIMIT = 20
DELAY_SECONDS = 20

# ===== MANUAL RESUME CONFIGURATION (for factory reset recovery) =====
# If Kaggle deletes your output file but you know which day you're on:
# 1. Set MANUAL_START_IMAGE to resume from specific image number
# 2. The cell will skip all images before that number
# 
# Day calculation: MANUAL_START_IMAGE = (day_number - 1) * IMAGE_LIMIT + 1
# Examples:
#   Day 1: MANUAL_START_IMAGE = None (or 1)
#   Day 2: MANUAL_START_IMAGE = 21  (skip first 20)
#   Day 3: MANUAL_START_IMAGE = 41  (skip first 40)
#   Day 4: MANUAL_START_IMAGE = 61  (skip first 60)
#   Day 5: MANUAL_START_IMAGE = 81  (skip first 80)
MANUAL_START_IMAGE = None  # Leave as None for automatic detection

# 1. Setup Gemini Client
try:
    user_secrets = UserSecretsClient()
    # Note: Corrected the secret label to be consistent with the print statement
    GEMINI_API_KEY = user_secrets.get_secret("gemini_api")
    print("API key retrieved successfully.")
except Exception as e:
    print(f"Error: {e}")
    print("Make sure you added the secret 'gemini-api' in Add-ons -> Secrets")
    GEMINI_API_KEY = None # Handle the missing key case

# Ensure client is only initialized if key is available
if not GEMINI_API_KEY:
    raise EnvironmentError("Failed to retrieve GEMINI_API_KEY. Check your Kaggle secrets setup.")

from google import genai
from google.genai import types
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_NAME = "gemini-2.5-flash"

# ----------------------------------------
# --- HELPER FUNCTIONS ---
# ----------------------------------------

def load_disease_knowledge(json_path):
    """Load disease knowledge from JSON file."""
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data
    except Exception as e:
        return None

def save_progress_state(processed_images, session_info):
    """Save processing progress to persistent file (survives kernel restart)."""
    state = {
        "processed_images": list(processed_images),
        "total_processed_count": len(processed_images),
        "last_updated": str(time.time()),
        "session_info": session_info
    }
    with open(PROGRESS_FILE, 'w', encoding='utf-8') as f:
        json.dump(state, f, indent=2)
    print(f"📋 Progress saved to {PROGRESS_FILE}")

def load_progress_state():
    """Load processing progress from persistent file."""
    if Path(PROGRESS_FILE).exists():
        try:
            with open(PROGRESS_FILE, 'r', encoding='utf-8') as f:
                state = json.load(f)
            processed = set(state.get("processed_images", []))
            print(f"📋 Loaded progress: {len(processed)} images previously processed")
            return processed, state.get("session_info", {})
        except Exception as e:
            print(f"⚠️  Could not load progress file: {e}")
            return set(), {}
    return set(), {}

def get_disease_context(disease_data):
    """Extract key medical context from disease JSON."""
    if not disease_data:
        return {}
    
    summaries = disease_data.get('section_summaries', {})
    context = {
        'disease_name': disease_data.get('disease_name', 'Unknown Disease'),
        'chapter_title': disease_data.get('chapter_title', ''),
        'epidemiology': summaries.get('epidemiology', 'N/A'),
        'clinical_manifestations': summaries.get('clinical_manifestations', 'N/A'),
        'diagnosis': summaries.get('diagnosis', 'N/A'),
        'treatment': summaries.get('treatment', 'N/A'),
        'pathogenesis': summaries.get('pathogenesis', 'N/A'),
    }
    return context

def find_matching_json(folder_name, json_dir):
    """Find matching JSON file for a disease folder."""
    json_dir_path = Path(json_dir)
    
    # Extract chapter number from folder name (e.g., "101_Malaria" -> "101")
    chapter_num = folder_name.split('_')[0]
    
    # Try exact match first
    for jf in json_dir_path.glob("*.json"):
        if jf.name.startswith(chapter_num + "_"):
            return jf
    
    # Try name-based matching
    json_file = json_dir_path / f"{folder_name}.json"
    if json_file.exists():
        return json_file
    
    return None

def generate_image_qa_pairs(image_filename, disease_context):
    """
    Generates 6 distinct Q&A pairs for a medical image.
    Covers visual analysis, clinical significance, and disease specifics.
    """
    
    # 1. Prepare Context
    title = disease_context.get('chapter_title', 'Unknown')
    disease_label = disease_context.get('disease_name', 'Unknown')
    epidemiology = disease_context.get('epidemiology', 'N/A')
    clinical_manifestations = disease_context.get('clinical_manifestations', 'N/A')
    diagnosis = disease_context.get('diagnosis', 'N/A')
    treatment = disease_context.get('treatment', 'N/A')
    pathogenesis = disease_context.get('pathogenesis', 'N/A')
    
    context_text = f"""
    Chapter Title: {title}
    Disease: {disease_label}
    
    Epidemiology: {epidemiology}
    Clinical Manifestations: {clinical_manifestations}
    Diagnosis: {diagnosis}
    Treatment: {treatment}
    Pathogenesis: {pathogenesis}
    """
    
    # 2. Advanced Prompt for Image-Based Q&A
    prompt = f"""
    You are an expert medical educator creating image-based fine-tuning dataset for disease recognition and medical analysis.
    
    CONTEXT (Medical Knowledge):
    {context_text}
    
    TASK:
    Generate exactly **3 distinct Instruction-Response pairs** based on a medical image showing {disease_label}.
    
    CRITICAL RULES FOR IMAGE ANALYSIS:
    - **Analyze the medical context first:** The image shows manifestations of {disease_label} based on the provided medical knowledge.
    - Generate visual analysis questions that incorporate the clinical, pathological, and epidemiological context.
    - Focus on multiple aspects: visual characteristics, clinical significance, diagnostic value, and management implications.
    - Each response must be derived strictly from the provided medical context.
    
    STRUCTURE FOR EACH PAIR:
    Each of the 6 pairs must be a self-contained object with:
    1. **"instruction"**: A question about visual characteristics, clinical significance, or diagnostic implications of the image
    2. **"response"**: The correct medical answer derived from the provided context and visual analysis
    
    REQUIRED QUESTION TYPES (Cover these 3 priorities):
    1. **Visual Description:** "Describe what you observe in this medical image. Focus on color, texture, morphology, and pattern."
    2. **Severity Assessment:** "Based on the visual findings, how would you assess the severity or stage of {disease_label}? What clinical implications does this have?"
    3. **Patient Management:** "Given these visual findings of {disease_label}, what would be your approach to patient management and monitoring?"
    
    CRITICAL REQUIREMENTS:
    - Each response must incorporate medical knowledge from the provided context
    - Responses should be detailed, educational, and clinically relevant
    - Focus on {disease_label} and its specific manifestations
    - Make responses comprehensive but concise (200-300 words per response)
    - Link visual observations to the clinical manifestations, diagnosis criteria, and treatment approaches
    
    OUTPUT FORMAT:
        Return a raw JSON list of 3 objects.
    Example:
    [
            {{"instruction": "Describe what you observe in this medical image...", "response": "..."}},
            {{"instruction": "Based on the visual findings, how would you assess severity...", "response": "..."}},
            {{"instruction": "Given these visual findings, what is your management plan...", "response": "..."}}
    ]
    """
    
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            )
        )
        return json.loads(response.text)
    except Exception as e:
        print(f"Error processing {title}: {e}")
        return []

# ----------------------------------------
# --- MAIN EXECUTION LOOP ---
# ----------------------------------------

all_image_qa_pairs = []
processed_image_files = set()  # Track which images have been processed

# ===== LOAD PERSISTENT PROGRESS (survives kernel restart & factory reset) =====
progress_file_processed, session_info = load_progress_state()

# Try to load from output JSON file if it exists
output_file_exists = Path(OUTPUT_FILE).exists()
if output_file_exists:
    try:
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            existing = json.load(f)
            if isinstance(existing, list):
                # Backfill missing disease_specific_focus using the source JSON's disease_name when possible
                for entry in existing:
                    if "disease_specific_focus" not in entry:
                        src_name = entry.get("source_file", "Unknown")
                        disease_from_json = "Unknown"
                        if isinstance(src_name, str):
                            candidate = Path(JSON_INPUT_DIR) / src_name
                            if candidate.exists():
                                try:
                                    with open(candidate, 'r', encoding='utf-8') as jf:
                                        jd = json.load(jf)
                                        disease_from_json = jd.get("disease_name", src_name.replace(".json", ""))
                                except Exception:
                                    disease_from_json = src_name.replace(".json", "")
                            else:
                                disease_from_json = src_name.replace(".json", "")
                        entry["disease_specific_focus"] = disease_from_json
                    # Track which images have been processed (from output file)
                    if "image_file" in entry:
                        processed_image_files.add(entry["image_file"])
                all_image_qa_pairs.extend(existing)
                print(f"✅ Loaded {len(existing)} existing pairs from {OUTPUT_FILE}.")
                print(f"✅ Identified {len(processed_image_files)} unique images from output file.")
    except Exception as e:
        print(f"⚠️  Could not load {OUTPUT_FILE}: {e}")
        output_file_exists = False

# ===== HANDLE FACTORY RESET / MANUAL RESUME =====
if not output_file_exists:
    if MANUAL_START_IMAGE is not None:
        # Manual override: user specifies which image to start from
        print(f"⚠️  OUTPUT FILE MISSING (factory reset detected)")
        print(f"🔧 MANUAL RESUME MODE: Starting from image {MANUAL_START_IMAGE}")
        # Skip first (MANUAL_START_IMAGE - 1) images in the processing loop
    elif len(progress_file_processed) > 0:
        # Use progress file as backup
        processed_image_files = progress_file_processed
        print(f"⚠️  OUTPUT FILE MISSING (factory reset detected)")
        print(f"📋 Using progress file: {len(processed_image_files)} images to skip")
    else:
        print(f"🆕 Starting fresh - no previous progress detected")

image_dir_path = Path(IMAGE_INPUT_DIR)
json_dir_path = Path(JSON_INPUT_DIR)

# Get all disease folders
image_files_list = []
for disease_folder in sorted(image_dir_path.iterdir()):
    if disease_folder.is_dir():
        # Convert generators to lists and combine them
        image_extensions = list(disease_folder.glob("*.png")) + list(disease_folder.glob("*.jpg")) + list(disease_folder.glob("*.jpeg"))
        for img in image_extensions:
            image_files_list.append((disease_folder.name, img))

print(f"\nFound {len(image_files_list)} total images in dataset.")
print(f"✅ Previously processed: {len(processed_image_files)} images")
print(f"📈 Processing strategy: Process {IMAGE_LIMIT} NEW images per run")
print(f"⏳ Remaining to process: {len(image_files_list) - len(processed_image_files)} images\n")

# 1. Iterate and Generate
images_processed_this_run = 0
images_skipped_count = 0

for i, (folder_name, image_path) in enumerate(image_files_list):
    # Calculate relative path for comparison
    image_relative_path = str(image_path.relative_to(image_dir_path))
    
    # ===== HANDLE MANUAL START (for factory reset recovery) =====
    if MANUAL_START_IMAGE is not None and (i + 1) < MANUAL_START_IMAGE:
        # Skip images before manual start point
        images_skipped_count += 1
        continue
    
    # Skip if already processed (from progress file or output file)
    if image_relative_path in processed_image_files:
        continue
    
    # Check if we've processed enough images in this run
    if images_processed_this_run >= IMAGE_LIMIT:
        print(f"\nReached the configured limit of {IMAGE_LIMIT} images for this run.")
        print(f"Progress: Processed {len(processed_image_files) + images_processed_this_run} / {len(image_files_list)} total images")
        print(f"Next run will continue from image {len(processed_image_files) + images_processed_this_run + 1}")
        break
    
    try:
        # Find matching JSON file
        json_file = find_matching_json(folder_name, JSON_INPUT_DIR)
        
        if json_file is None or not json_file.exists():
            continue
        
        # Load disease knowledge
        disease_data = load_disease_knowledge(json_file)
        if not disease_data:
            continue
        
        disease_context = get_disease_context(disease_data)
        base_file_name = os.path.basename(json_file)
        
        # Verify image can be opened
        img = Image.open(image_path)
        img.verify()
        
        d_name = disease_context.get('disease_name', 'Unknown')
        print(f"[Day Progress: {images_processed_this_run+1}/{IMAGE_LIMIT}] Generating 3 pairs for: {d_name} from image {image_path.name}...")
        
        # Generate Q&A pairs for this image using Gemini API
        pairs = generate_image_qa_pairs(image_path.name, disease_context)
        
        if pairs:
            # Keep only the top 3 pairs in case the model returns more
            pairs = pairs[:3]
            modified_pairs = []
            # Add provenance so each pair is traceable to its image and source JSON
            for pair in pairs:
                pair["source_file"] = base_file_name
                pair["image_file"] = str(image_path.relative_to(image_dir_path))
                pair["disease_specific_focus"] = d_name
                modified_pairs.append(pair)
            
            all_image_qa_pairs.extend(modified_pairs)
            print(f"   -> Success. Generated {len(modified_pairs)} pairs. Total gathered: {len(all_image_qa_pairs)}")
            images_processed_this_run += 1  # Only increment on successful processing
        else:
            print("   -> No pairs returned.")
        
    except Exception as e:
        print(f"Skipping file due to error: {e}")

    # 2. Add the wait time (between successful API calls)
    if images_processed_this_run < IMAGE_LIMIT and images_processed_this_run > 0:
        print(f"   -> Waiting {DELAY_SECONDS} seconds before starting the next image...")
        time.sleep(DELAY_SECONDS)

# ----------------------------------------
# 3. SAVE TO PRETTY-PRINTED JSON ARRAY
# ----------------------------------------
print(f"\nSaving all gathered data ({len(all_image_qa_pairs)} entries) to a pretty-printed JSON file...")

# Key Change: Dump the entire list, using 'indent=4' for readability
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(all_image_qa_pairs, f, indent=4, ensure_ascii=False)

# ===== SAVE PERSISTENT PROGRESS STATE (survives kernel restart) =====
updated_processed = processed_image_files.copy()
for pair in all_image_qa_pairs:
    if "image_file" in pair:
        updated_processed.add(pair["image_file"])

session_info = {
    "last_run_images": images_processed_this_run,
    "total_pairs_generated": len(all_image_qa_pairs),
    "completion_percentage": round((len(updated_processed) / len(image_files_list)) * 100, 1) if len(image_files_list) > 0 else 0
}
save_progress_state(updated_processed, session_info)

print(f"\n{'='*60}")
print(f"✅ SESSION COMPLETE!")
print(f"{'='*60}")
print(f"   Generated this run: {images_processed_this_run} images × 3 pairs = {images_processed_this_run * 3} pairs")
print(f"   Total dataset size: {len(all_image_qa_pairs)} pairs")
print(f"   Overall progress: {len(updated_processed)} / {len(image_files_list)} images processed")
print(f"   Completion: {session_info['completion_percentage']}%")
print(f"   Remaining: {len(image_files_list) - len(updated_processed)} images")
print(f"\n📊 Output files:")
print(f"   • Dataset: {OUTPUT_FILE}")
print(f"   • Progress: {PROGRESS_FILE} (survives kernel restart)")
print(f"\n💡 Next run will automatically skip {len(updated_processed)} processed images")
print(f"{'='*60}")
